# 03 - Strategy / Engine Separation

Demonstrates the modular workflow:

    raw ticks -> M15 BID candles -> strategy -> normalized signals
    -> reusable tick execution engine -> trade log

The strategy lives in `src/strategies/candle_direction.py` and the execution
engine in `src/backtest.py`. Neither module is duplicated in this notebook.

The result must reproduce the golden reference from `02_backtest_core.ipynb`.

## Imports and constants

In [1]:
import os
import sys
import inspect

import numpy as np
import pandas as pd

# make the project root importable (this notebook runs from notebooks/)
sys.path.insert(0, os.path.abspath(".."))

from src.strategies.candle_direction import generate_signals
from src.backtest import run_backtest

DATA_PATH = "../data/XAUUSD_DUKAS_2025_01_ticks.csv"
SYMBOL = "XAUUSD"
TIMEFRAME = "15min"

BACKTEST_START_UTC = "2025-01-31 00:00:00+00:00"
BACKTEST_END_UTC   = "2025-01-31 22:00:00+00:00"

STOP_DISTANCE = 5.0
TAKE_PROFIT_DISTANCE = 10.0

SOURCE_COLUMNS = ["date", "time", "bid", "ask", "bid_volume", "ask_volume"]

## Load ticks (READ-ONLY)

Same source, same range, same UTC parsing as the golden notebook.

In [2]:
start = pd.Timestamp(BACKTEST_START_UTC)
end = pd.Timestamp(BACKTEST_END_UTC)

raw = pd.read_csv(DATA_PATH, header=None, names=SOURCE_COLUMNS)
raw["timestamp"] = pd.to_datetime(
    raw["date"] + " " + raw["time"],
    format="%Y.%m.%d %H:%M:%S.%f",
).dt.tz_localize("UTC")

ticks = raw.loc[(raw["timestamp"] >= start) & (raw["timestamp"] <= end), ["timestamp", "bid", "ask"]].reset_index(drop=True)
del raw

print("Ticks in range:", len(ticks))
print("First tick:", ticks["timestamp"].iloc[0])
print("Last  tick:", ticks["timestamp"].iloc[-1])

Ticks in range: 322548
First tick: 2025-01-31 00:00:00.044000+00:00
Last  tick: 2025-01-31 21:59:56.320000+00:00


## Build M15 BID candles

Same construction as the golden notebook: 15min, label=left, closed=left.

In [3]:
def build_candles(ticks, timeframe=TIMEFRAME):
    bid = ticks.set_index("timestamp")["bid"]
    return bid.resample(timeframe, label="left", closed="left").ohlc().dropna(how="all")


candles = build_candles(ticks)
print("M15 candles:", len(candles))

M15 candles: 88


## Strategy -> normalized signals

The strategy module receives candles and returns normalized signals only.

In [4]:
signals = generate_signals(candles, timeframe=TIMEFRAME)

print("Signals:", len(signals))
print("By direction:", signals["direction"].value_counts().to_dict())
display(signals.head())

Signals: 88
By direction: {'LONG': 45, 'SHORT': 43}


,signal_candle_start,signal_available_time,direction
0,2025-01-31 00:00:00+00:00,2025-01-31 00:15:00+00:00,LONG
1,2025-01-31 00:15:00+00:00,2025-01-31 00:30:00+00:00,LONG
2,2025-01-31 00:30:00+00:00,2025-01-31 00:45:00+00:00,SHORT
3,2025-01-31 00:45:00+00:00,2025-01-31 01:00:00+00:00,SHORT
4,2025-01-31 01:00:00+00:00,2025-01-31 01:15:00+00:00,LONG


## Reusable engine -> trade log

The engine receives normalized signals and real ticks. It does not know how the
signals were generated.

In [5]:
trades = run_backtest(ticks, signals, STOP_DISTANCE, TAKE_PROFIT_DISTANCE)

print("Trades:", len(trades))
display(trades)

Trades: 7


,trade_id,direction,signal_candle_start,signal_available_time,entry_time,entry_bid,entry_ask,entry_price,stop_loss,take_profit,exit_time,exit_bid,exit_ask,exit_price,exit_reason,pnl_price,duration_seconds,spread_at_entry,spread_at_exit
0,1,LONG,2025-01-31 00:00:00+00:00,2025-01-31 00:15:00+00:00,2025-01-31 00:15:01.237000+00:00,2797.47,2798.07,2798.07,2793.07,2808.07,2025-01-31 06:55:24.853000+00:00,2792.91,2793.38,2792.91,SL,-5.16,24023.616,0.60,0.47
1,2,SHORT,2025-01-31 06:45:00+00:00,2025-01-31 07:00:00+00:00,2025-01-31 07:00:00.037000+00:00,2792.68,2793.16,2792.68,2797.68,2782.68,2025-01-31 07:20:03.286000+00:00,2797.16,2797.70,2797.70,SL,-5.02,1203.249,0.48,0.54
2,3,LONG,2025-01-31 07:15:00+00:00,2025-01-31 07:30:00+00:00,2025-01-31 07:30:00.071000+00:00,2799.49,2800.41,2800.41,2795.41,2810.41,2025-01-31 08:22:03.135000+00:00,2795.39,2795.97,2795.39,SL,-5.02,3123.064,0.92,0.58
3,4,SHORT,2025-01-31 08:15:00+00:00,2025-01-31 08:30:00+00:00,2025-01-31 08:30:00.114000+00:00,2793.39,2794.09,2793.39,2798.39,2783.39,2025-01-31 11:40:07.759000+00:00,2797.45,2798.43,2798.43,SL,-5.04,11407.645,0.70,0.98
4,5,LONG,2025-01-31 11:30:00+00:00,2025-01-31 11:45:00+00:00,2025-01-31 11:45:00.098000+00:00,2796.88,2797.78,2797.78,2792.78,2807.78,2025-01-31 13:40:01.514000+00:00,2807.82,2808.28,2807.82,TP,10.04,6901.416,0.90,0.46
5,6,LONG,2025-01-31 13:30:00+00:00,2025-01-31 13:45:00+00:00,2025-01-31 13:45:00.118000+00:00,2807.05,2807.68,2807.68,2802.68,2817.68,2025-01-31 18:18:40.593000+00:00,2802.50,2802.91,2802.50,SL,-5.18,16420.475,0.63,0.41
6,7,SHORT,2025-01-31 18:15:00+00:00,2025-01-31 18:30:00+00:00,2025-01-31 18:30:00.011000+00:00,2799.57,2800.05,2799.57,2804.57,2789.57,2025-01-31 21:59:56.320000+00:00,2798.09,2800.01,2800.01,END_OF_DATA,-0.44,12596.309,0.48,1.92


## Golden reference

Expected trade log taken from the already verified `02_backtest_core.ipynb` run.
Used only for regression comparison.

In [6]:
REFERENCE_ROWS = [
    {"trade_id": 1, "direction": "LONG", "signal_candle_start": "2025-01-31T00:00:00+00:00", "signal_available_time": "2025-01-31T00:15:00+00:00", "entry_time": "2025-01-31T00:15:01.237000+00:00", "entry_bid": 2797.47, "entry_ask": 2798.07, "entry_price": 2798.07, "stop_loss": 2793.07, "take_profit": 2808.07, "exit_time": "2025-01-31T06:55:24.853000+00:00", "exit_bid": 2792.91, "exit_ask": 2793.38, "exit_price": 2792.91, "exit_reason": "SL", "pnl_price": -5.160000000000309, "duration_seconds": 24023.616, "spread_at_entry": 0.6000000000003638, "spread_at_exit": 0.47000000000025466},
    {"trade_id": 2, "direction": "SHORT", "signal_candle_start": "2025-01-31T06:45:00+00:00", "signal_available_time": "2025-01-31T07:00:00+00:00", "entry_time": "2025-01-31T07:00:00.037000+00:00", "entry_bid": 2792.68, "entry_ask": 2793.16, "entry_price": 2792.68, "stop_loss": 2797.68, "take_profit": 2782.68, "exit_time": "2025-01-31T07:20:03.286000+00:00", "exit_bid": 2797.16, "exit_ask": 2797.7, "exit_price": 2797.7, "exit_reason": "SL", "pnl_price": -5.019999999999982, "duration_seconds": 1203.249, "spread_at_entry": 0.4800000000000182, "spread_at_exit": 0.5399999999999636},
    {"trade_id": 3, "direction": "LONG", "signal_candle_start": "2025-01-31T07:15:00+00:00", "signal_available_time": "2025-01-31T07:30:00+00:00", "entry_time": "2025-01-31T07:30:00.071000+00:00", "entry_bid": 2799.49, "entry_ask": 2800.41, "entry_price": 2800.41, "stop_loss": 2795.41, "take_profit": 2810.41, "exit_time": "2025-01-31T08:22:03.135000+00:00", "exit_bid": 2795.39, "exit_ask": 2795.97, "exit_price": 2795.39, "exit_reason": "SL", "pnl_price": -5.019999999999982, "duration_seconds": 3123.064, "spread_at_entry": 0.9200000000000728, "spread_at_exit": 0.5799999999999272},
    {"trade_id": 4, "direction": "SHORT", "signal_candle_start": "2025-01-31T08:15:00+00:00", "signal_available_time": "2025-01-31T08:30:00+00:00", "entry_time": "2025-01-31T08:30:00.114000+00:00", "entry_bid": 2793.39, "entry_ask": 2794.09, "entry_price": 2793.39, "stop_loss": 2798.39, "take_profit": 2783.39, "exit_time": "2025-01-31T11:40:07.759000+00:00", "exit_bid": 2797.45, "exit_ask": 2798.43, "exit_price": 2798.43, "exit_reason": "SL", "pnl_price": -5.039999999999964, "duration_seconds": 11407.645, "spread_at_entry": 0.7000000000002728, "spread_at_exit": 0.9800000000000182},
    {"trade_id": 5, "direction": "LONG", "signal_candle_start": "2025-01-31T11:30:00+00:00", "signal_available_time": "2025-01-31T11:45:00+00:00", "entry_time": "2025-01-31T11:45:00.098000+00:00", "entry_bid": 2796.88, "entry_ask": 2797.78, "entry_price": 2797.78, "stop_loss": 2792.78, "take_profit": 2807.78, "exit_time": "2025-01-31T13:40:01.514000+00:00", "exit_bid": 2807.82, "exit_ask": 2808.28, "exit_price": 2807.82, "exit_reason": "TP", "pnl_price": 10.039999999999964, "duration_seconds": 6901.416, "spread_at_entry": 0.900000000000091, "spread_at_exit": 0.4600000000000364},
    {"trade_id": 6, "direction": "LONG", "signal_candle_start": "2025-01-31T13:30:00+00:00", "signal_available_time": "2025-01-31T13:45:00+00:00", "entry_time": "2025-01-31T13:45:00.118000+00:00", "entry_bid": 2807.05, "entry_ask": 2807.68, "entry_price": 2807.68, "stop_loss": 2802.68, "take_profit": 2817.68, "exit_time": "2025-01-31T18:18:40.593000+00:00", "exit_bid": 2802.5, "exit_ask": 2802.91, "exit_price": 2802.5, "exit_reason": "SL", "pnl_price": -5.179999999999836, "duration_seconds": 16420.475, "spread_at_entry": 0.6299999999996544, "spread_at_exit": 0.4099999999998545},
    {"trade_id": 7, "direction": "SHORT", "signal_candle_start": "2025-01-31T18:15:00+00:00", "signal_available_time": "2025-01-31T18:30:00+00:00", "entry_time": "2025-01-31T18:30:00.011000+00:00", "entry_bid": 2799.57, "entry_ask": 2800.05, "entry_price": 2799.57, "stop_loss": 2804.57, "take_profit": 2789.57, "exit_time": "2025-01-31T21:59:56.320000+00:00", "exit_bid": 2798.09, "exit_ask": 2800.01, "exit_price": 2800.01, "exit_reason": "END_OF_DATA", "pnl_price": -0.44000000000005457, "duration_seconds": 12596.309, "spread_at_entry": 0.4800000000000182, "spread_at_exit": 1.9200000000000728},
]

reference = pd.DataFrame(REFERENCE_ROWS)
for col in ["signal_candle_start", "signal_available_time", "entry_time", "exit_time"]:
    reference[col] = pd.to_datetime(reference[col], utc=True)

display(reference)

,trade_id,direction,signal_candle_start,signal_available_time,entry_time,entry_bid,entry_ask,entry_price,stop_loss,take_profit,exit_time,exit_bid,exit_ask,exit_price,exit_reason,pnl_price,duration_seconds,spread_at_entry,spread_at_exit
0,1,LONG,2025-01-31 00:00:00+00:00,2025-01-31 00:15:00+00:00,2025-01-31 00:15:01.237000+00:00,2797.47,2798.07,2798.07,2793.07,2808.07,2025-01-31 06:55:24.853000+00:00,2792.91,2793.38,2792.91,SL,-5.16,24023.616,0.60,0.47
1,2,SHORT,2025-01-31 06:45:00+00:00,2025-01-31 07:00:00+00:00,2025-01-31 07:00:00.037000+00:00,2792.68,2793.16,2792.68,2797.68,2782.68,2025-01-31 07:20:03.286000+00:00,2797.16,2797.70,2797.70,SL,-5.02,1203.249,0.48,0.54
2,3,LONG,2025-01-31 07:15:00+00:00,2025-01-31 07:30:00+00:00,2025-01-31 07:30:00.071000+00:00,2799.49,2800.41,2800.41,2795.41,2810.41,2025-01-31 08:22:03.135000+00:00,2795.39,2795.97,2795.39,SL,-5.02,3123.064,0.92,0.58
3,4,SHORT,2025-01-31 08:15:00+00:00,2025-01-31 08:30:00+00:00,2025-01-31 08:30:00.114000+00:00,2793.39,2794.09,2793.39,2798.39,2783.39,2025-01-31 11:40:07.759000+00:00,2797.45,2798.43,2798.43,SL,-5.04,11407.645,0.70,0.98
4,5,LONG,2025-01-31 11:30:00+00:00,2025-01-31 11:45:00+00:00,2025-01-31 11:45:00.098000+00:00,2796.88,2797.78,2797.78,2792.78,2807.78,2025-01-31 13:40:01.514000+00:00,2807.82,2808.28,2807.82,TP,10.04,6901.416,0.90,0.46
5,6,LONG,2025-01-31 13:30:00+00:00,2025-01-31 13:45:00+00:00,2025-01-31 13:45:00.118000+00:00,2807.05,2807.68,2807.68,2802.68,2817.68,2025-01-31 18:18:40.593000+00:00,2802.50,2802.91,2802.50,SL,-5.18,16420.475,0.63,0.41
6,7,SHORT,2025-01-31 18:15:00+00:00,2025-01-31 18:30:00+00:00,2025-01-31 18:30:00.011000+00:00,2799.57,2800.05,2799.57,2804.57,2789.57,2025-01-31 21:59:56.320000+00:00,2798.09,2800.01,2800.01,END_OF_DATA,-0.44,12596.309,0.48,1.92


## Regression check

Field-by-field, order-sensitive comparison of the modular trade log against the
golden reference. Price and timestamp fields are compared exactly; PnL is
compared exactly as well because the engine reproduces the identical float path.

In [7]:
def pf(value):
    return "PASS" if value else "FAIL"


def series_equal(a, b):
    a = pd.Series(np.asarray(a)).reset_index(drop=True)
    b = pd.Series(np.asarray(b)).reset_index(drop=True)
    if len(a) != len(b):
        return False
    return bool((a == b).all())


count_pass = len(trades) == len(reference)
dir_pass = series_equal(trades["direction"], reference["direction"])
sig_start_pass = series_equal(pd.to_datetime(trades["signal_candle_start"]), reference["signal_candle_start"])
sig_avail_pass = series_equal(pd.to_datetime(trades["signal_available_time"]), reference["signal_available_time"])
entry_time_pass = series_equal(pd.to_datetime(trades["entry_time"]), reference["entry_time"])
entry_bid_pass = series_equal(trades["entry_bid"], reference["entry_bid"])
entry_ask_pass = series_equal(trades["entry_ask"], reference["entry_ask"])
entry_price_pass = series_equal(trades["entry_price"], reference["entry_price"])
sl_pass = series_equal(trades["stop_loss"], reference["stop_loss"])
tp_pass = series_equal(trades["take_profit"], reference["take_profit"])
exit_time_pass = series_equal(pd.to_datetime(trades["exit_time"]), reference["exit_time"])
exit_bid_pass = series_equal(trades["exit_bid"], reference["exit_bid"])
exit_ask_pass = series_equal(trades["exit_ask"], reference["exit_ask"])
exit_price_pass = series_equal(trades["exit_price"], reference["exit_price"])
exit_reason_pass = series_equal(trades["exit_reason"], reference["exit_reason"])
pnl_pass = series_equal(trades["pnl_price"], reference["pnl_price"])
trade_id_pass = series_equal(trades["trade_id"], reference["trade_id"])
duration_pass = series_equal(trades["duration_seconds"], reference["duration_seconds"])
spread_entry_pass = series_equal(trades["spread_at_entry"], reference["spread_at_entry"])
spread_exit_pass = series_equal(trades["spread_at_exit"], reference["spread_at_exit"])

print("GOLDEN TRADE COUNT:", pf(count_pass))
print("GOLDEN TRADE DIRECTIONS:", pf(dir_pass))
print("GOLDEN ENTRY TIMES:", pf(entry_time_pass))
print("GOLDEN ENTRY PRICES:", pf(entry_price_pass))
print("GOLDEN EXIT TIMES:", pf(exit_time_pass))
print("GOLDEN EXIT PRICES:", pf(exit_price_pass))
print("GOLDEN EXIT REASONS:", pf(exit_reason_pass))
print("GOLDEN PNL:", pf(pnl_pass))
print("GOLDEN TRADE IDS:", pf(trade_id_pass))
print("GOLDEN DURATION:", pf(duration_pass))
print("GOLDEN SPREAD AT ENTRY:", pf(spread_entry_pass))
print("GOLDEN SPREAD AT EXIT:", pf(spread_exit_pass))

full_regression = all([
    count_pass, dir_pass, sig_start_pass, sig_avail_pass,
    entry_time_pass, entry_bid_pass, entry_ask_pass, entry_price_pass,
    sl_pass, tp_pass, exit_time_pass, exit_bid_pass, exit_ask_pass,
    exit_price_pass, exit_reason_pass, pnl_pass,
    trade_id_pass, duration_pass, spread_entry_pass, spread_exit_pass,
])
print("FULL REGRESSION:", pf(full_regression))

engine_params = list(inspect.signature(run_backtest).parameters)
separation_pass = engine_params == ["ticks", "signals", "stop_distance", "take_profit_distance"]
print("ENGINE/STRATEGY SEPARATION:", pf(separation_pass))

GOLDEN TRADE COUNT: PASS
GOLDEN TRADE DIRECTIONS: PASS
GOLDEN ENTRY TIMES: PASS
GOLDEN ENTRY PRICES: PASS
GOLDEN EXIT TIMES: PASS
GOLDEN EXIT PRICES: PASS
GOLDEN EXIT REASONS: PASS
GOLDEN PNL: PASS
GOLDEN TRADE IDS: PASS
GOLDEN DURATION: PASS
GOLDEN SPREAD AT ENTRY: PASS
GOLDEN SPREAD AT EXIT: PASS
FULL REGRESSION: PASS
ENGINE/STRATEGY SEPARATION: PASS
